In [3]:
import sys


In [2]:
import os
import re
import numpy as np
import pandas as pd
import Levenshtein
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [3]:
# Create required output directory
os.makedirs("output", exist_ok=True)

# Dataset paths
from pathlib import Path

DATASET_DIR = Path(r"C:\Users\chinm\Downloads\dataset\student_resource\dataset")
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

# Load Training Data
train_s1 = pd.read_csv(TRAIN_DIR / "train_source1.tsv", sep="\t")
train_s2 = pd.read_csv(TRAIN_DIR / "train_source2.tsv", sep="\t")
train_s3 = pd.read_csv(TRAIN_DIR / "train_source3.tsv", sep="\t")
train_gt = pd.read_csv(TRAIN_DIR / "train_ground_truth.tsv", sep="\t")

# Load Test Data
test_s1 = pd.read_csv(TEST_DIR / "test_source1.tsv", sep="\t")
test_s2 = pd.read_csv(TEST_DIR / "test_source2.tsv", sep="\t")
test_s3 = pd.read_csv(TEST_DIR / "test_source3.tsv", sep="\t")

In [4]:
def clean_text_pure_python(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'\\b(corporation|corp)\\b', 'corp', text)
    text = re.sub(r'\\b(private|pvt)\\b', 'pvt', text)
    text = re.sub(r'\\b(limited|ltd)\\b', 'ltd', text)
    text = re.sub(r'\\s+and\\s+', ' & ', text)
    text = re.sub(r'[^a-z0-9\\s&]', '', text)
    return re.sub(r'\\s+', ' ', text).strip()

def clean_series_safe(series, batch_size=50000):
    cleaned_list = []
    total_len = len(series)
    for start_idx in range(0, total_len, batch_size):
        batch = series.iloc[start_idx:start_idx + batch_size]
        cleaned_batch = [clean_text_pure_python(x) for x in batch]
        cleaned_list.extend(cleaned_batch)
    return cleaned_list

for df in [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]:
    print(f"Cleaning dataframe securely with PyArrow...")
    name_list = clean_series_safe(df['business_name'])
    addr_list = clean_series_safe(df['business_address'])
    df['clean_name'] = pd.Series(name_list, dtype="string[pyarrow]")
    df['clean_address'] = pd.Series(addr_list, dtype="string[pyarrow]")

print("Text preprocessing completed successfully!")


Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Cleaning dataframe securely with PyArrow...
Text preprocessing completed successfully!


In [5]:
import os

# Create an output folder for cleaned data if it doesn't exist
os.makedirs("output/cleaned_data", exist_ok=True)

# Dictionary of dataframes to export
cleaned_dfs = {
    "train_source1": train_s1,
    "train_source2": train_s2,
    "train_source3": train_s3,
    "test_source1": test_s1,
    "test_source2": test_s2,
    "test_source3": test_s3
}

# Export each dataframe to a tab-separated (.tsv) file
for name, df in cleaned_dfs.items():
    output_path = f"output/cleaned_data/{name}_cleaned.tsv"
    df.to_csv(output_path, sep="\t", index=False)
    print(f"Successfully exported: {output_path}")

print("All cleaned databases exported successfully!")

Successfully exported: output/cleaned_data/train_source1_cleaned.tsv
Successfully exported: output/cleaned_data/train_source2_cleaned.tsv
Successfully exported: output/cleaned_data/train_source3_cleaned.tsv
Successfully exported: output/cleaned_data/test_source1_cleaned.tsv
Successfully exported: output/cleaned_data/test_source2_cleaned.tsv
Successfully exported: output/cleaned_data/test_source3_cleaned.tsv
All cleaned databases exported successfully!


In [2]:
for df in [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]:
    print("Cleaning dataframe...")
    df['clean_name'] = clean_series_safe(df['business_name'])       # plain object dtype, no pyarrow round-trip
    df['clean_address'] = clean_series_safe(df['business_address'])

print("Text preprocessing completed successfully!")

NameError: name 'train_s2' is not defined

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


def generate_candidates(s1_df, s2_df, s3_df, threshold=0.20, top_k=15):
    print("Starting candidate generation...")
    candidate_pairs = []

    s1_df = s1_df.copy()
    s2_df = s2_df.copy()
    s3_df = s3_df.copy()

    print("Preparing country blocking keys...")
    fill_val = "__UNKNOWN__"
    s1_df['_country_key'] = s1_df['country'].fillna(fill_val)
    s2_df['_country_key'] = s2_df['country'].fillna(fill_val)
    s3_df['_country_key'] = s3_df['country'].fillna(fill_val)

    countries = s1_df['_country_key'].unique()
    print(f"Processing {len(countries)} country blocks...")

    for country in countries:
        s1_sub = s1_df[s1_df['_country_key'] == country]
        if s1_sub.empty:
            continue

        if country == fill_val:
            pool_df = pd.concat([s2_df, s3_df], ignore_index=True)
        else:
            s2_sub = s2_df[(s2_df['_country_key'] == country) | (s2_df['_country_key'] == fill_val)]
            s3_sub = s3_df[(s3_df['_country_key'] == country) | (s3_df['_country_key'] == fill_val)]
            pool_df = pd.concat([s2_sub, s3_sub], ignore_index=True)

        if pool_df.empty:
            continue

        pool_names = pool_df['clean_name'].fillna('').astype(str)
        s1_names = s1_sub['clean_name'].fillna('').astype(str)
        if not pool_names.str.strip().any():
            print(f"Skipping country block with no usable names: {country}")
            continue

        print(f"Generating TF-IDF candidates for country block: {country}")
        vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 4))
        tfidf_pool = vectorizer.fit_transform(pool_names)
        tfidf_s1 = vectorizer.transform(s1_names)

        k = min(top_k, tfidf_pool.shape[0])
        print(f"Searching the top {k} nearest neighbors...")
        nn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute')
        nn.fit(tfidf_pool)
        distances, indices = nn.kneighbors(tfidf_s1)

        similarities = 1 - distances

        s1_ids = s1_sub['entity_id'].to_numpy()
        pool_ids = pool_df['entity_id'].to_numpy()

        for i in range(len(s1_ids)):
            row_sims = similarities[i]
            row_idxs = indices[i]
            keep_mask = row_sims >= threshold
            for j, sim in zip(row_idxs[keep_mask], row_sims[keep_mask]):
                candidate_pairs.append({
                    'source1_entity_id': s1_ids[i],
                    'candidate_entity_id': pool_ids[j],
                    'sim_score': sim
                })

    print(f"Candidate generation complete: {len(candidate_pairs)} pairs retained.")
    return pd.DataFrame(candidate_pairs)


def check_blocking_recall(candidates_df, gt_df):
    print("Checking blocking recall...")
    gt_dict = {}
    for _, row in gt_df.iterrows():
        matches = str(row['matched_entity_ids']).split(',') if pd.notna(row['matched_entity_ids']) else []
        gt_dict[str(row['source1_entity_id'])] = set(str(m).strip() for m in matches if str(m).strip())

    candidates_df = candidates_df.copy()
    candidates_df['source1_entity_id'] = candidates_df['source1_entity_id'].astype(str)
    candidates_df['candidate_entity_id'] = candidates_df['candidate_entity_id'].astype(str)

    cand_dict = candidates_df.groupby('source1_entity_id')['candidate_entity_id'].apply(set).to_dict()

    total_gt_matches = 0
    recovered = 0
    fully_missed_entities = 0

    for s1_id, true_matches in gt_dict.items():
        if not true_matches:
            continue
        candidate_set = cand_dict.get(s1_id, set())
        hit = true_matches & candidate_set
        total_gt_matches += len(true_matches)
        recovered += len(hit)
        if not hit:
            fully_missed_entities += 1

    recall = recovered / total_gt_matches if total_gt_matches else 0.0
    print(f"Blocking recall: {recovered}/{total_gt_matches} = {recall:.4f}")
    print(f"Entities with ZERO surviving true match: {fully_missed_entities}")
    return recall

required_dataframes = ["train_s1", "train_s2", "train_s3", "train_gt"]
missing_data = any(name not in globals() for name in required_dataframes)
missing_clean_names = not missing_data and any(
    "clean_name" not in globals()[name].columns
    for name in ["train_s1", "train_s2", "train_s3"]
)

if missing_data or missing_clean_names:
    print("Training data is not initialized. Loading the prepared training files...")
    cleaned_dir_candidates = [
        Path("notebooks/joe/output/cleaned_data"),
        Path("output/cleaned_data"),
    ]
    cleaned_dir = next((path for path in cleaned_dir_candidates if path.exists()), None)
    if cleaned_dir is None:
        raise FileNotFoundError(
            "Prepared files were not found. Checked: "
            + ", ".join(str(path.resolve()) for path in cleaned_dir_candidates)
        )

    dataset_dir = Path(r"C:\Users\chinm\Downloads\dataset\student_resource\dataset")
    train_dir = dataset_dir / "train"

    train_s1 = pd.read_csv(cleaned_dir / "train_source1_cleaned.tsv", sep="\t")
    train_s2 = pd.read_csv(cleaned_dir / "train_source2_cleaned.tsv", sep="\t")
    train_s3 = pd.read_csv(cleaned_dir / "train_source3_cleaned.tsv", sep="\t")
    train_gt = pd.read_csv(train_dir / "train_ground_truth.tsv", sep="\t")
    print(f"Prepared training files loaded from: {cleaned_dir.resolve()}")

print("Generating training candidates and evaluating blocking performance...")

print("Generating training candidates and evaluating blocking performance...")

SAMPLE_MODE = True   # flip to False when ready for the full run

if SAMPLE_MODE:
    sample_s1 = train_s1.sample(n=5000, random_state=42)
    sample_s2 = train_s2.sample(n=min(20000, len(train_s2)), random_state=42)
    sample_s3 = train_s3.sample(n=min(20000, len(train_s3)), random_state=42)
    train_candidates = generate_candidates(sample_s1, sample_s2, sample_s3, threshold=0.20, top_k=15)
else:
    train_candidates = generate_candidates(train_s1, train_s2, train_s3, threshold=0.20, top_k=15)

check_blocking_recall(train_candidates, train_gt)

Generating training candidates and evaluating blocking performance...
Generating training candidates and evaluating blocking performance...
Starting candidate generation...
Preparing country blocking keys...
Processing 2 country blocks...
Generating TF-IDF candidates for country block: US
Searching the top 15 nearest neighbors...
Generating TF-IDF candidates for country block: India
Searching the top 15 nearest neighbors...
Candidate generation complete: 68540 pairs retained.
Checking blocking recall...


KeyboardInterrupt: 